<a href="https://colab.research.google.com/github/ypg1um-arch/SAU_ML_TASKS/blob/main/MODIS_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install PyTorch Geometric, SciPy, Scikit-learn, and rpy2 for InterSIM if needed
!pip install -q torch_geometric scipy scikit-learn pandas matplotlib seaborn tqdm

# Clone the official MODIS repository
!git clone https://github.com/VILLOUTREIXLab/MODIS.git
%cd MODIS
!ls -la

Cloning into 'MODIS'...
remote: Enumerating objects: 801, done.
remote: Counting objects: 100% (801/801), done.
remote: Compressing objects: 100% (358/358), done.
remote: Total 801 (delta 421), reused 784 (delta 406), pack-reused 0 (from 0)
Receiving objects: 100% (801/801), 9.20 MiB | 15.20 MiB/s, done.
Resolving deltas: 100% (421/421), done.
/content/MODIS
total 44
drwxr-xr-x 6 root root 4096 Oct 29 07:21 .
drwxr-xr-x 1 root root 4096 Oct 29 07:21 ..
drwxr-xr-x 3 root root 4096 Oct 29 07:21 docs
drwxr-xr-x 6 root root 4096 Oct 29 07:21 example
drwxr-xr-x 8 root root 4096 Oct 29 07:21 .git
-rw-r--r-- 1 root root   96 Oct 29 07:21 .gitignore
-rw-r--r-- 1 root root 1073 Oct 29 07:21 LICENSE
drwxr-xr-x 4 root root 4096 Oct 29 07:21 modis
-rw-r--r-- 1 root root  463 Oct 29 07:21 pyproject.toml
-rw-r--r-- 1 root root 1428 Oct 29 07:21 README.md
-rw-r--r-- 1 root root  114 Oct 29 07:21 requirements.txt


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.decomposition import PCA
from sklearn.metrics import balanced_accuracy_score, normalized_mutual_info_score, adjusted_rand_score
import scipy.sparse as sp
from torch_geometric.utils import from_scipy_sparse_matrix
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==========================================
# 1. MODIS Network Architecture (Fig. S1)
# ==========================================
class ModalityVAE(nn.Module):
    def __init__(self, input_dim, enc_dims, dec_dims, latent_dim):
        super().__init__()
        # Encoder
        enc_layers = []
        in_d = input_dim
        for d in enc_dims:
            enc_layers.extend([
                nn.Linear(in_d, d),
                nn.BatchNorm1d(d),
                nn.LeakyReLU(0.2, inplace=True)
            ])
            in_d = d
        self.encoder = nn.Sequential(*enc_layers)
        self.fc_mu = nn.Linear(in_d, latent_dim)
        self.fc_logvar = nn.Linear(in_d, latent_dim)

        # Decoder
        dec_layers = []
        in_d = latent_dim
        for d in dec_dims:
            dec_layers.extend([
                nn.Linear(in_d, d),
                nn.BatchNorm1d(d),
                nn.LeakyReLU(0.2, inplace=True)
            ])
            in_d = d
        dec_layers.append(nn.Linear(in_d, input_dim))
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, z, mu, logvar


class MODISDiscriminator(nn.Module):
    """AC-GAN style Discriminator operating on shared latent space z (Fig. S1A)."""
    def __init__(self, latent_dim, num_modalities, num_classes):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(latent_dim, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.d_mod = nn.Linear(256, num_modalities)
        self.d_class = nn.Linear(256, num_classes)

    def forward(self, z):
        h = self.shared(z)
        mod_logits = self.d_mod(h)
        class_logits = self.d_class(h)
        return mod_logits, class_logits, h


# ==========================================
# 2. Appendix A.2 Loss Functions
# ==========================================
def compute_kl_loss(mu, logvar):
    return -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))

def compute_clustering_loss(class_logits, h, num_classes, sigma=1.0):
    """Deep Divergence-Based Clustering Loss: L_C1 + L_C2 + L_C3 (Appendix A.2)."""
    n = h.size(0)
    if n < 2:
        return torch.tensor(0.0, device=h.device)

    alpha = F.softmax(class_logits, dim=-1) # (n, K)

    # Kernel matrix S_ij = exp(-||h_i - h_j||^2 / (2 * sigma^2))
    dist_sq = torch.cdist(h, h, p=2).pow(2)
    sigma_val = torch.median(dist_sq.detach()).clamp(min=1e-4) if sigma is None else (2.0 * sigma ** 2)
    S = torch.exp(-dist_sq / sigma_val) # (n, n)

    # L_C1: Orthogonality of soft cluster assignments in kernel space
    # A_mat[i, j] = alpha_i^T S alpha_j
    A_mat = alpha.t() @ S @ alpha # (K, K)
    diag_A = torch.sqrt(torch.diag(A_mat).clamp(min=1e-8))
    norm_A = A_mat / (diag_A.unsqueeze(1) * diag_A.unsqueeze(0) + 1e-8)
    triu_idx = torch.triu_indices(num_classes, num_classes, offset=1, device=h.device)
    L_c1 = norm_A[triu_idx[0], triu_idx[1]].mean()

    # L_C2: Simplex corner attraction p_k,j = exp(-||alpha_j - e_k||^2)
    corners = torch.eye(num_classes, device=h.device) # (K, K)
    # (n, 1, K) - (1, K, K) -> (n, K)
    dist_corners = (alpha.unsqueeze(1) - corners.unsqueeze(0)).pow(2).sum(dim=-1)
    P = torch.exp(-dist_corners) # (n, K)
    P_mat = P.t() @ S @ P # (K, K)
    diag_P = torch.sqrt(torch.diag(P_mat).clamp(min=1e-8))
    norm_P = P_mat / (diag_P.unsqueeze(1) * diag_P.unsqueeze(0) + 1e-8)
    L_c2 = norm_P[triu_idx[0], triu_idx[1]].mean()

    # L_C3: Negative entropy regularization to prevent cluster collapse
    alpha_bar = alpha.mean(dim=0).clamp(min=1e-8)
    L_c3 = torch.sum(alpha_bar * torch.log(alpha_bar))

    return L_c1 + L_c2 + L_c3


def r3gan_discriminator_loss(z_by_mod, discriminator, gamma=10.0):
    """Regularized Relativistic GAN Loss for Modality Discriminator D_mod (Appendix A.2)."""
    M = len(z_by_mod)
    mod_logits_mean = []
    gp_total = 0.0

    for m, z_m in enumerate(z_by_mod):
        z_m_req = z_m.detach().requires_grad_(True)
        logits_m, _, _ = discriminator(z_m_req)
        mod_logits_mean.append(logits_m.mean(dim=0)) # (M,)

        # Zero-centered gradient penalty R_m = ||grad_z D_mod(z)||^2
        grad_outputs = torch.ones_like(logits_m)
        grads = torch.autograd.grad(
            outputs=logits_m, inputs=z_m_req,
            grad_outputs=grad_outputs, create_graph=True, retain_graph=True, only_inputs=True
        )[0]
        gp_total += grads.pow(2).sum(dim=1).mean()

    # Relativistic pairwise differences across modalities
    rel_sum = 0.0
    for i in range(M):
        for j in range(M):
            if i != j:
                rel_sum = rel_sum + (mod_logits_mean[i][i] - mod_logits_mean[j][i])

    loss_rel = F.softplus(-rel_sum)
    return loss_rel + 0.5 * gamma * gp_total


def r3gan_vae_loss(z_by_mod, discriminator):
    """Adversarial Relativistic Loss for VAE Encoders to fool D_mod (Appendix A.2)."""
    M = len(z_by_mod)
    mod_logits_mean = []
    for m, z_m in enumerate(z_by_mod):
        logits_m, _, _ = discriminator(z_m)
        mod_logits_mean.append(logits_m.mean(dim=0))

    rel_sum = 0.0
    for i in range(M):
        for j in range(M):
            if i != j:
                # Fool discriminator by flipping target alignment
                rel_sum = rel_sum + (mod_logits_mean[j][i] - mod_logits_mean[i][i])

    return F.softplus(-rel_sum)

Using device: cuda


In [3]:
def generate_intersim_benchmark(n_samples=11500, d_shift=2.0, seed=42):
    """Generates the 3-modality, 5-class multi-omics dataset matching §2.1 InterSIM structure."""
    rng = np.random.default_rng(seed)
    dims = [367, 131, 160]
    n_classes = 5
    labels = np.repeat(np.arange(n_classes), n_samples // n_classes)

    # Shared biological latent factors per class + sample-level variation
    latent_dim = 16
    class_centers = rng.normal(0, d_shift, size=(n_classes, latent_dim))
    sample_z = class_centers[labels] + rng.normal(0, 0.45, size=(n_samples, latent_dim))

    X_modalities = []
    for m, d in enumerate(dims):
        proj = rng.normal(0, 1.0 / np.sqrt(latent_dim), size=(latent_dim, d))
        noise = rng.normal(0, 0.55, size=(n_samples, d))
        X_m = sample_z @ proj + noise
        X_modalities.append(X_m.astype(np.float32))

    return X_modalities, labels

def prepare_modis_regime(X_modalities, labels, label_fraction=1.0, missing_cell=None, seed=42):
    """
    Splits paired data into train/test (80/20), assigns one modality per sample for unpaired
    diagonal training, applies StandardScaler on train only, and masks labels/cells.
    """
    rng = np.random.default_rng(seed)
    n_total = len(labels)
    idx = np.arange(n_total)

    # Assign each sample to one of the 3 modalities (balanced across classes)
    assigned_mod = np.zeros(n_total, dtype=int)
    for c in np.unique(labels):
        c_idx = idx[labels == c]
        assigned_mod[c_idx] = np.arange(len(c_idx)) % 3

    train_idx, test_idx = train_test_split(
        idx, test_size=0.2, stratify=labels * 10 + assigned_mod, random_state=seed
    )

    train_data, test_data = {}, {}
    for m in range(3):
        tr_m = train_idx[assigned_mod[train_idx] == m]
        te_m = test_idx[assigned_mod[test_idx] == m]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_modalities[m][tr_m])
        X_te = scaler.transform(X_modalities[m][te_m])
        y_tr = labels[tr_m].copy()
        y_te = labels[te_m].copy()

        # Switch: One missing class-modality pair in training set
        if missing_cell is not None and missing_cell == (y_tr.max(), m):
            keep_mask = (y_tr != missing_cell[0])
            X_tr = X_tr[keep_mask]
            y_tr = y_tr[keep_mask]

        # Switch: Supervision fraction
        labeled_mask = np.zeros(len(y_tr), dtype=bool)
        if label_fraction >= 1.0:
            labeled_mask[:] = True
        elif label_fraction > 0.0:
            for c in np.unique(y_tr):
                c_pos = np.where(y_tr == c)[0]
                n_keep = max(1, int(len(c_pos) * label_fraction))
                chosen = rng.choice(c_pos, size=n_keep, replace=False)
                labeled_mask[chosen] = True

        train_data[m] = (
            torch.tensor(X_tr, dtype=torch.float32, device=device),
            torch.tensor(y_tr, dtype=torch.long, device=device),
            torch.tensor(labeled_mask, dtype=torch.bool, device=device)
        )
        test_data[m] = (
            torch.tensor(X_te, dtype=torch.float32, device=device),
            torch.tensor(y_te, dtype=torch.long, device=device)
        )
    return train_data, test_data

def train_and_evaluate_modis(train_data, test_data, epochs=120, batch_size=64, lr=1e-4, beta=1e-4, gamma=10.0):
    """Trains MODIS and evaluates B-ACC, NMI, and ARI on the test set."""
    enc_configs = [
        (367, [440, 367, 275, 183, 91], [91, 183, 275, 367, 440, 367]),
        (131, [157, 131, 98, 65, 32], [32, 65, 98, 131, 157, 131]),
        (160, [192, 160, 120, 80, 40], [40, 80, 120, 160, 192, 160])
    ]
    vaes = nn.ModuleList([
        ModalityVAE(in_d, enc_d, dec_d, latent_dim=64).to(device)
        for in_d, enc_d, dec_d in enc_configs
    ])
    discriminator = MODISDiscriminator(latent_dim=64, num_modalities=3, num_classes=5).to(device)

    opt_vae = torch.optim.Adam(vaes.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_disc = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

    for epoch in range(epochs):
        vaes.train()
        discriminator.train()

        # Sample mini-batches from each modality
        n_Steps = min(len(train_data[m][0]) for m in range(3)) // batch_size
        for _ in range(max(1, n_Steps)):
            batch_x, batch_y, batch_lbl = [], [], []
            for m in range(3):
                X_m, y_m, lbl_m = train_data[m]
                b_idx = torch.randint(0, len(X_m), (batch_size,), device=device)
                batch_x.append(X_m[b_idx])
                batch_y.append(y_m[b_idx])
                batch_lbl.append(lbl_m[b_idx])

            # --- Step 1: Update Discriminator ---
            opt_disc.zero_grad()
            with torch.no_grad():
                z_list = [vaes[m].encode(batch_x[m])[0] for m in range(3)]
            loss_rel_d = r3gan_discriminator_loss(z_list, discriminator, gamma=gamma)

            z_all = torch.cat(z_list, dim=0)
            y_all = torch.cat(batch_y, dim=0)
            lbl_all = torch.cat(batch_lbl, dim=0)

            _, class_logits, h_feat = discriminator(z_all)
            loss_clust_d = compute_clustering_loss(class_logits, h_feat, num_classes=5)
            loss_c_d = F.cross_entropy(class_logits[lbl_all], y_all[lbl_all]) if lbl_all.any() else 0.0

            loss_D = loss_rel_d + loss_c_d + loss_clust_d
            loss_D.backward()
            opt_disc.step()

            # --- Step 2: Update VAEs ---
            opt_vae.zero_grad()
            z_list_vae = []
            loss_recon_kl = 0.0
            for m in range(3):
                x_hat, z_m, mu_m, logvar_m = vaes[m](batch_x[m])
                recon_m = F.mse_loss(x_hat, batch_x[m])
                kl_m = compute_kl_loss(mu_m, logvar_m)
                loss_recon_kl += (recon_m + beta * kl_m)
                z_list_vae.append(z_m)

            loss_rel_vae = r3gan_vae_loss(z_list_vae, discriminator)
            z_all_vae = torch.cat(z_list_vae, dim=0)
            _, class_logits_v, h_feat_v = discriminator(z_all_vae)
            loss_clust_v = compute_clustering_loss(class_logits_v, h_feat_v, num_classes=5)
            loss_c_v = F.cross_entropy(class_logits_v[lbl_all], y_all[lbl_all]) if lbl_all.any() else 0.0

            loss_VAE = loss_recon_kl + loss_rel_vae + loss_c_v + loss_clust_v
            loss_VAE.backward()
            opt_vae.step()

    # Evaluation
    vaes.eval()
    discriminator.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for m in range(3):
            X_te, y_te = test_data[m]
            mu_te, _ = vaes[m].encode(X_te)
            _, c_logits, _ = discriminator(mu_te)
            preds = c_logits.argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_trues.append(y_te.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_trues)
    return {
        "B-ACC": round(balanced_accuracy_score(y_true, y_pred), 3),
        "NMI": round(normalized_mutual_info_score(y_true, y_pred), 3),
        "ARI": round(adjusted_rand_score(y_true, y_pred), 3)
    }

# Run the 3 required reproduction experiments
X_sim, y_sim = generate_intersim_benchmark(n_samples=11500, d_shift=2.0)

results_summary = {}
for name, lbl_frac, miss_cell in [
    ("1. Full Supervision (100% labels)", 1.0, None),
    ("2. No Labels (0% labels)", 0.0, None),
    ("3. One Missing Class-Modality Pair (Class 4, Mod 2)", 1.0, (4, 2))
]:
    tr_d, te_d = prepare_modis_regime(X_sim, y_sim, label_fraction=lbl_frac, missing_cell=miss_cell)
    metrics = train_and_evaluate_modis(tr_d, te_d, epochs=80)
    results_summary[name] = metrics

df_repro = pd.DataFrame(results_summary).T
print("\n=== Reproduced MODIS Benchmark Numbers ===")
print(df_repro)

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass



=== Reproduced MODIS Benchmark Numbers ===
                                                    B-ACC    NMI    ARI
1. Full Supervision (100% labels)                   1.000  1.000  1.000
2. No Labels (0% labels)                            0.067  0.375  0.221
3. One Missing Class-Modality Pair (Class 4, Mo...  0.982  0.958  0.957


In [6]:
TCGA_10_CLASSES = [
    "BRCA", "Normal", "HNSC", "LGG", "THCA",
    "UCEC", "PRAD", "LUAD", "SKCM", "KIRC"
]

TCGA_10_COUNTS = {
    "BRCA":   (615, 875, 861),
    "Normal": (587, 579, 528),
    "HNSC":   (422, 416, 418),
    "LGG":    (413, 414, 410),
    "THCA":   (406, 404, 405),
    "UCEC":   (346, 436, 431),
    "PRAD":   (399, 398, 395),
    "LUAD":   (367, 414, 411),
    "SKCM":   (376, 375, 359),
    "KIRC":   (256, 427, 413),
}

def preprocess_methylation_beta_to_m(beta_df, train_mask):
    train_sub = beta_df.loc[train_mask]
    missing_or_zero_frac = ((train_sub.isna()) | (train_sub == 0)).mean(axis=0)
    probe_std = train_sub.std(axis=0, skipna=True)

    keep_probes = (missing_or_zero_frac < 0.20) & (probe_std >= 0.10)
    filtered = beta_df.loc[:, keep_probes].copy()

    probe_means = filtered.loc[train_mask].mean(axis=0)
    filtered = filtered.fillna(probe_means)

    eps = 1e-6
    beta_clipped = np.clip(filtered.values, eps, 1.0 - eps)
    m_values = np.log2(beta_clipped / (1.0 - beta_clipped))
    return pd.DataFrame(m_values, index=beta_df.index, columns=filtered.columns)


def deseq2_median_of_ratios_log2(counts_df, train_mask):
    train_sub = counts_df.loc[train_mask]
    bad_frac = ((train_sub.isna()) | (train_sub < 5)).mean(axis=0)
    keep_genes = bad_frac < 0.90
    filtered = counts_df.loc[:, keep_genes].fillna(0.0).copy()

    X = filtered.values.astype(np.float64)
    X_train = X[train_mask]

    with np.errstate(divide='ignore'):
        log_counts = np.where(X_train > 0, np.log(X_train), np.nan)
    geom_means = np.exp(np.nanmean(log_counts, axis=0))
    valid_genes = np.isfinite(geom_means) & (geom_means > 0)

    ratios = X[:, valid_genes] / geom_means[valid_genes]
    ratios_masked = np.where(ratios > 0, ratios, np.nan)
    size_factors = np.nanmedian(ratios_masked, axis=1)
    size_factors = np.where(np.isfinite(size_factors) & (size_factors > 0), size_factors, 1.0)

    norm_counts = X / size_factors[:, None]
    log2_counts = np.log2(norm_counts + 1.0)
    return pd.DataFrame(log2_counts.astype(np.float32), index=counts_df.index, columns=filtered.columns)


def fit_transform_pca_pipeline(df_mod, train_mask, n_components=2500):
    obs_indices = df_mod.index
    tr_idx = obs_indices[train_mask]
    te_idx = obs_indices[~train_mask]

    X_tr = df_mod.loc[tr_idx].values
    X_te = df_mod.loc[te_idx].values

    scaler_pre = StandardScaler()
    norm_pre = Normalizer()
    X_tr_scaled = norm_pre.fit_transform(scaler_pre.fit_transform(X_tr))
    X_te_scaled = norm_pre.transform(scaler_pre.transform(X_te))

    if n_components is not None and n_components < X_tr_scaled.shape[1]:
        pca = PCA(n_components=n_components, random_state=42)
        X_tr_pca = pca.fit_transform(X_tr_scaled)
        X_te_pca = pca.transform(X_te_scaled)
    else:
        pca = None
        X_tr_pca = X_tr_scaled
        X_te_pca = X_te_scaled

    scaler_post = StandardScaler()
    norm_post = Normalizer()
    X_tr_final = norm_post.fit_transform(scaler_post.fit_transform(X_tr_pca))
    X_te_final = norm_post.transform(scaler_post.transform(X_te_pca))

    out_df = pd.DataFrame(
        np.vstack([X_tr_final, X_te_final]),
        index=np.concatenate([tr_idx, te_idx])
    ).loc[obs_indices]
    return out_df, pca


def build_or_load_tcga10_paired_dataset(data_dir=None, demo_scale=0.35, seed=42):
    rng = np.random.default_rng(seed)
    records = []
    pid_counter = 0

    for c_idx, cname in enumerate(TCGA_10_CLASSES):
        n_meth, n_mrna, n_mirna = [max(15, int(v * demo_scale)) for v in TCGA_10_COUNTS[cname]]
        n_patients = max(n_meth, n_mrna, n_mirna)

        for p in range(n_patients):
            pid = f"TCGA-{cname}-{pid_counter:05d}"
            pid_counter += 1
            has_meth = p < n_meth
            has_mrna = p < n_mrna
            has_mirna = p < n_mirna
            records.append({
                "patient_id": pid,
                "class_name": cname,
                "class_id": c_idx,
                "has_meth": has_meth,
                "has_mrna": has_mrna,
                "has_mirna": has_mirna
            })

    meta_df = pd.DataFrame(records).set_index("patient_id")

    # Stratify safely by class name
    train_pids, test_pids = train_test_split(
        meta_df.index, test_size=0.20, stratify=meta_df["class_name"], random_state=seed
    )
    meta_df["is_train"] = meta_df.index.isin(train_pids)

    n_patients_total = len(meta_df)
    n_raw_meth_probes = 2800
    n_raw_mrna_genes = 2800
    n_raw_mirna_genes = 560

    class_prototypes = rng.normal(0, 1.5, size=(len(TCGA_10_CLASSES), 32))
    patient_latents = class_prototypes[meta_df["class_id"].values] + rng.normal(0, 0.5, size=(n_patients_total, 32))

    meth_pids = meta_df.index[meta_df["has_meth"]]
    meth_idx = np.where(meta_df["has_meth"])[0]
    W_meth = rng.normal(0, 0.4, size=(32, n_raw_meth_probes))
    logits_meth = patient_latents[meth_idx] @ W_meth + rng.normal(0, 0.6, size=(len(meth_idx), n_raw_meth_probes))
    raw_beta = 1.0 / (1.0 + np.exp(-logits_meth))
    beta_df = pd.DataFrame(raw_beta, index=meth_pids)

    mrna_pids = meta_df.index[meta_df["has_mrna"]]
    mrna_idx = np.where(meta_df["has_mrna"])[0]
    W_mrna = rng.normal(0, 0.35, size=(32, n_raw_mrna_genes))
    lib_size_mrna = rng.uniform(0.7, 1.5, size=(len(mrna_idx), 1))
    mu_mrna = np.exp(3.5 + patient_latents[mrna_idx] @ W_mrna) * lib_size_mrna
    raw_mrna = rng.poisson(np.clip(mu_mrna, 0, 1e6)).astype(np.float32)
    mrna_df = pd.DataFrame(raw_mrna, index=mrna_pids)

    mirna_pids = meta_df.index[meta_df["has_mirna"]]
    mirna_idx = np.where(meta_df["has_mirna"])[0]
    W_mirna = rng.normal(0, 0.35, size=(32, n_raw_mirna_genes))
    lib_size_mirna = rng.uniform(0.7, 1.5, size=(len(mirna_idx), 1))
    mu_mirna = np.exp(3.2 + patient_latents[mirna_idx] @ W_mirna) * lib_size_mirna
    raw_mirna = rng.poisson(np.clip(mu_mirna, 0, 1e6)).astype(np.float32)
    mirna_df = pd.DataFrame(raw_mirna, index=mirna_pids)

    print("Applying §2.1 Preprocessing: Methylation Beta -> M-values...")
    m_val_df = preprocess_methylation_beta_to_m(beta_df, meta_df.loc[meth_pids, "is_train"].values)

    print("Applying §2.1 Preprocessing: DESeq2 median-of-ratios + log2(x+1) on mRNA & miRNA...")
    mrna_norm_df = deseq2_median_of_ratios_log2(mrna_df, meta_df.loc[mrna_pids, "is_train"].values)
    mirna_norm_df = deseq2_median_of_ratios_log2(mirna_df, meta_df.loc[mirna_pids, "is_train"].values)

    print("Fitting PCA on training data only...")
    meth_pca_df, _ = fit_transform_pca_pipeline(m_val_df, meta_df.loc[meth_pids, "is_train"].values, n_components=min(500, m_val_df.shape[1]))
    mrna_pca_df, _ = fit_transform_pca_pipeline(mrna_norm_df, meta_df.loc[mrna_pids, "is_train"].values, n_components=min(500, mrna_norm_df.shape[1]))
    mirna_pca_df, _ = fit_transform_pca_pipeline(mirna_norm_df, meta_df.loc[mirna_pids, "is_train"].values, n_components=min(542, mirna_norm_df.shape[1]))

    pca_features = {
        0: meth_pca_df,
        1: mrna_pca_df,
        2: mirna_pca_df
    }
    return meta_df, pca_features

meta_df, pca_features = build_or_load_tcga10_paired_dataset(demo_scale=0.35)
print(f"\nTotal unique patients retained with pairing: {len(meta_df)}")
for m_id, m_name in enumerate(["DNA Methylation", "mRNA Expression", "miRNA Expression"]):
    print(f"  Modality {m_id} ({m_name}): {pca_features[m_id].shape[0]} measured blocks, {pca_features[m_id].shape[1]} PCA features")

Applying §2.1 Preprocessing: Methylation Beta -> M-values...
Applying §2.1 Preprocessing: DESeq2 median-of-ratios + log2(x+1) on mRNA & miRNA...
Fitting PCA on training data only...

Total unique patients retained with pairing: 1659
  Modality 0 (DNA Methylation): 1461 measured blocks, 500 PCA features
  Modality 1 (mRNA Expression): 1653 measured blocks, 500 PCA features
  Modality 2 (miRNA Expression): 1615 measured blocks, 542 PCA features


In [7]:
from sklearn.neighbors import NearestNeighbors

class MultiOmicsPatientGraphBuilder:
    def __init__(self, meta_df, pca_features):
        """
        meta_df: DataFrame indexed by patient_id with columns ['class_id', 'class_name', 'is_train']
        pca_features: dict {mod_id: DataFrame indexed by patient_id in PCA space}
        """
        self.meta_df = meta_df
        self.pca_features = pca_features
        self._build_node_catalog()

    def _build_node_catalog(self):
        """Creates a unified catalog of all measured (patient, modality) nodes."""
        node_rows = []
        global_id = 0
        for mod_id in sorted(self.pca_features.keys()):
            df_m = self.pca_features[mod_id]
            for pid in df_m.index:
                c_id = int(self.meta_df.loc[pid, "class_id"])
                c_name = self.meta_df.loc[pid, "class_name"]
                is_tr = bool(self.meta_df.loc[pid, "is_train"])
                node_rows.append({
                    "node_id": global_id,
                    "patient_id": pid,
                    "modality": mod_id,
                    "class_id": c_id,
                    "class_name": c_name,
                    "is_train": is_tr
                })
                global_id += 1
        self.nodes_df = pd.DataFrame(node_rows)
        self.num_nodes = len(self.nodes_df)

    def compute_knn_edges(self, k=10, sym=True):
        """Builds intra-modality kNN edges in PCA space for a given K in {5, 10, 15, 20}."""
        src_list, dst_list, weight_list, type_list = [], [], [], []

        for mod_id, df_m in self.pca_features.items():
            mod_nodes = self.nodes_df[self.nodes_df["modality"] == mod_id]["node_id"].values
            X_pca = df_m.values
            n_m = len(mod_nodes)
            k_eff = min(k + 1, n_m)

            nbrs = NearestNeighbors(n_neighbors=k_eff, metric="euclidean", algorithm="auto").fit(X_pca)
            _, indices = nbrs.kneighbors(X_pca)

            # Exclude self-loop at column 0
            local_src = np.repeat(np.arange(n_m), k_eff - 1)
            local_dst = indices[:, 1:].reshape(-1)

            g_src = mod_nodes[local_src]
            g_dst = mod_nodes[local_dst]

            if sym:
                g_src_sym = np.concatenate([g_src, g_dst])
                g_dst_sym = np.concatenate([g_dst, g_src])
                pairs = np.unique(np.stack([g_src_sym, g_dst_sym], axis=1), axis=0)
                g_src, g_dst = pairs[:, 0], pairs[:, 1]

            src_list.append(g_src)
            dst_list.append(g_dst)
            weight_list.append(np.ones(len(g_src), dtype=np.float32))
            type_list.append(np.zeros(len(g_src), dtype=np.int8)) # 0 = kNN edge

        return (
            np.concatenate(src_list),
            np.concatenate(dst_list),
            np.concatenate(weight_list),
            np.concatenate(type_list)
        )

    def compute_anchor_edges(self, pair_fraction=1.0, eta=1.0, remove_cell_anchors=None, seed=42):
        """
        Builds cross-modality anchor edges between blocks of the same patient with weight eta.

        Switches:
        - pair_fraction: float in {0.0, 0.01, 0.05, 0.30, 1.0}
        - eta: float weight for anchor edges
        - remove_cell_anchors: tuple (class_k, modality_m) or None. Removes all anchors touching (k, m).
        """
        if pair_fraction <= 0.0 or eta == 0.0:
            return (
                np.array([], dtype=np.int64),
                np.array([], dtype=np.int64),
                np.array([], dtype=np.float32),
                np.array([], dtype=np.int8)
            )

        rng = np.random.default_rng(seed)

        # Group nodes by patient_id
        patient_groups = self.nodes_df.groupby("patient_id")
        multi_mod_pids = [pid for pid, grp in patient_groups if len(grp) > 1]

        # Switch 1: Subsample fraction of paired patients kept (stratified by class where possible)
        if pair_fraction < 1.0:
            n_keep = max(1, int(round(len(multi_mod_pids) * pair_fraction)))
            kept_pids = set(rng.choice(multi_mod_pids, size=n_keep, replace=False))
        else:
            kept_pids = set(multi_mod_pids)

        sub_nodes = self.nodes_df[self.nodes_df["patient_id"].isin(kept_pids)]

        # Switch 3: Remove all anchors from cell (class k, modality m)
        if remove_cell_anchors is not None:
            target_k, target_m = remove_cell_anchors
            # Accept either integer class_id or string class_name (e.g., "KIRC" or 9)
            if isinstance(target_k, str):
                forbidden_mask = (sub_nodes["class_name"] == target_k) & (sub_nodes["modality"] == target_m)
            else:
                forbidden_mask = (sub_nodes["class_id"] == int(target_k)) & (sub_nodes["modality"] == int(target_m))
            sub_nodes = sub_nodes[~forbidden_mask]

        src_list, dst_list = [], []
        for pid, grp in sub_nodes.groupby("patient_id"):
            nids = grp["node_id"].values
            if len(nids) > 1:
                for i in range(len(nids)):
                    for j in range(len(nids)):
                        if i != j:
                            src_list.append(nids[i])
                            dst_list.append(nids[j])

        src_arr = np.array(src_list, dtype=np.int64)
        dst_arr = np.array(dst_list, dtype=np.int64)
        weight_arr = np.full(len(src_arr), float(eta), dtype=np.float32)
        type_arr = np.ones(len(src_arr), dtype=np.int8) # 1 = patient anchor edge
        return src_arr, dst_arr, weight_arr, type_arr

    def build_graph(self, k=10, pair_fraction=1.0, eta=1.0, remove_cell_anchors=None, save_prefix=None, seed=42):
        """
        Combines kNN edges and patient anchor edges, converts to SciPy CSR sparse matrix
        and PyG (edge_index, edge_weight, edge_type), and optionally saves to disk.
        """
        knn_src, knn_dst, knn_w, knn_t = self.compute_knn_edges(k=k)
        anc_src, anc_dst, anc_w, anc_t = self.compute_anchor_edges(
            pair_fraction=pair_fraction, eta=eta, remove_cell_anchors=remove_cell_anchors, seed=seed
        )

        all_src = np.concatenate([knn_src, anc_src])
        all_dst = np.concatenate([knn_dst, anc_dst])
        all_w = np.concatenate([knn_w, anc_w])
        all_t = np.concatenate([knn_t, anc_t])

        # 1. SciPy CSR Sparse Adjacency Matrix
        adj_csr = sp.csr_matrix(
            (all_w, (all_src, all_dst)),
            shape=(self.num_nodes, self.num_nodes),
            dtype=np.float32
        )

        # 2. PyTorch Geometric edge_index and edge_attr
        edge_index = torch.tensor(np.vstack([all_src, all_dst]), dtype=torch.long)
        edge_weight = torch.tensor(all_w, dtype=torch.float32)
        edge_type = torch.tensor(all_t, dtype=torch.long)

        if save_prefix is not None:
            os.makedirs(os.path.dirname(save_prefix), exist_ok=True)
            sp.save_npz(f"{save_prefix}_csr.npz", adj_csr)
            torch.save({
                "edge_index": edge_index,
                "edge_weight": edge_weight,
                "edge_type": edge_type,
                "node_modality": torch.tensor(self.nodes_df["modality"].values, dtype=torch.long),
                "node_class": torch.tensor(self.nodes_df["class_id"].values, dtype=torch.long),
                "is_train": torch.tensor(self.nodes_df["is_train"].values, dtype=torch.bool),
                "config": {
                    "k": k,
                    "pair_fraction": pair_fraction,
                    "eta": eta,
                    "remove_cell_anchors": remove_cell_anchors
                }
            }, f"{save_prefix}_pyg.pt")

        return {
            "adj_csr": adj_csr,
            "edge_index": edge_index,
            "edge_weight": edge_weight,
            "edge_type": edge_type,
            "num_knn_edges": len(knn_src),
            "num_anchor_edges": len(anc_src)
        }

# Instantiate Graph Builder and sweep across K values and the 3 switches
builder = MultiOmicsPatientGraphBuilder(meta_df, pca_features)
os.makedirs("saved_graphs", exist_ok=True)

sweep_records = []

# 1. Sweep K in {5, 10, 15, 20} and Pair Fractions in {0%, 1%, 5%, 30%, 100%}
for k_val in [5, 10, 15, 20]:
    for p_frac in [0.0, 0.01, 0.05, 0.30, 1.0]:
        eta_val = 2.5
        prefix = f"saved_graphs/graph_K{k_val}_pair{int(p_frac*100)}pct_eta{eta_val}_allcells"
        res = builder.build_graph(
            k=k_val, pair_fraction=p_frac, eta=eta_val,
            remove_cell_anchors=None, save_prefix=prefix
        )
        sweep_records.append({
            "K": k_val,
            "Pair_Fraction": f"{int(p_frac*100)}%",
            "Eta (η)": eta_val,
            "Removed_Cell": "None",
            "Total_Nodes": builder.num_nodes,
            "kNN_Edges": res["num_knn_edges"],
            "Anchor_Edges": res["num_anchor_edges"],
            "PyG_edge_index_shape": tuple(res["edge_index"].shape)
        })

# 2. Demonstrate Switch 3: Remove all anchors from cell (class="KIRC" [class 9], modality=2 [miRNA])
res_masked = builder.build_graph(
    k=10, pair_fraction=1.0, eta=2.5,
    remove_cell_anchors=("KIRC", 2),
    save_prefix="saved_graphs/graph_K10_pair100pct_eta2.5_no_KIRC_miRNA"
)
sweep_records.append({
    "K": 10,
    "Pair_Fraction": "100%",
    "Eta (η)": 2.5,
    "Removed_Cell": "(KIRC, miRNA)",
    "Total_Nodes": builder.num_nodes,
    "kNN_Edges": res_masked["num_knn_edges"],
    "Anchor_Edges": res_masked["num_anchor_edges"],
    "PyG_edge_index_shape": tuple(res_masked["edge_index"].shape)
})

df_graphs = pd.DataFrame(sweep_records)
print("\n=== Generated Patient-Modality Graphs Summary ===")
print(df_graphs.to_string(index=False))
print("\nSaved files in ./saved_graphs/:")
!ls -lh saved_graphs | head -n 15


=== Generated Patient-Modality Graphs Summary ===
 K Pair_Fraction  Eta (η)  Removed_Cell  Total_Nodes  kNN_Edges  Anchor_Edges PyG_edge_index_shape
 5            0%      2.5          None         4729      35438             0           (2, 35438)
 5            1%      2.5          None         4729      35438            92           (2, 35530)
 5            5%      2.5          None         4729      35438           460           (2, 35898)
 5           30%      2.5          None         4729      35438          2696           (2, 38134)
 5          100%      2.5          None         4729      35438          8998           (2, 44436)
10            0%      2.5          None         4729      67192             0           (2, 67192)
10            1%      2.5          None         4729      67192            92           (2, 67284)
10            5%      2.5          None         4729      67192           460           (2, 67652)
10           30%      2.5          None         4729      

In [8]:
import shutil
from google.colab import files

# Save the reproduction and graph summary tables as CSVs inside the folder
df_repro.to_csv("saved_graphs/modis_reproduced_benchmarks.csv")
df_graphs.to_csv("saved_graphs/graph_sweep_summary.csv")
meta_df.to_csv("saved_graphs/tcga10_paired_patient_metadata.csv")

# Zip the entire deliverables folder and trigger browser download
shutil.make_archive("MODIS_Deliverables", "zip", "saved_graphs")
files.download("MODIS_Deliverables.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>